# 2_work.ipynb — The Dirty Ledger (cleaning on a copy)

**Disclosure:** this notebook was written with AI assistance (Claude) following the instructor's instruction to use AI for the whole assignment. The raw file is never modified; all cleaning happens on a copy.

In [1]:
import pandas as pd, hashlib
pd.set_option('display.width',200); pd.set_option('display.max_columns',20)
path='teaching_prices_messy.csv'
print('sha256:',hashlib.sha256(open(path,'rb').read()).hexdigest())
raw=pd.read_csv(path)          # untouched original
df=raw.copy()                  # all cleaning happens on this copy
print(raw.shape); raw.head()

sha256: 5818dad3807487d4e37f764e5d19cbaa152edd78fe4db182eb28b087c8709d35
(97, 6)


,date,item,city,price,unit,data_label
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


## 1. Diagnosis (before changing anything)

In [2]:
print(raw.dtypes,'\n'); print(raw.isna().sum(),'\n')
print('exact duplicate rows:',raw.duplicated().sum())
print(raw.item.value_counts(),'\n'); print(raw.city.value_counts(),'\n')
print('non-numeric price strings:',raw.price[raw.price.str.contains(r'[^0-9.]',na=False)].tolist())
print('unique units:',raw.unit.nunique(),'| unique data_label:',raw.data_label.nunique())
p=pd.to_numeric(raw.price.str.replace(',','',regex=False),errors='coerce')
print(raw.assign(p=p).groupby('item').p.describe())

date          str
item          str
city          str
price         str
unit          str
data_label    str
dtype: object 

date          0
item          0
city          0
price         1
unit          0
data_label    0
dtype: int64 

exact duplicate rows: 1
item
Staple A       24
Oil B          24
Pulse C        24
Sweetener D    24
oil b           1
Name: count, dtype: int64 

city
City 1    49
City 2    48
Name: count, dtype: int64 

non-numeric price strings: ['1,250']
unique units: 1 | unique data_label: 1
             count        mean          std    min     25%     50%      75%     max
item                                                                               
Oil B         24.0  324.833333    14.529331  298.0  314.75  324.00  337.000   350.0
Pulse C       23.0  205.913043     3.912835  200.0  202.50  206.00  208.500   213.5
Staple A      24.0  158.333333   232.616809   99.5  106.25  112.50  116.125  1250.0
Sweetener D   24.0  576.812500  2006.939937  149.0  161.00  167

In [3]:
# problem rows, with neighbours for context
print(raw[raw.price.isna()|raw.price.isin(['1,250','9999'])|(raw.item=='oil b')|raw.duplicated(keep=False)][['date','item','city','price']])
# structure check: expect 12 dates x 4 items x 2 cities = 96
print('rows:',len(raw),'expected 96')
print(raw[(raw.date=='2026-02-05')&(raw.city=='City 1')][['date','item','city','price']])

          date         item    city  price
5   2026-01-01      Pulse C  City 2    NaN
8   2026-01-08     Staple A  City 1   99.5
17  2026-01-15     Staple A  City 2  1,250
30  2026-01-22  Sweetener D  City 1   9999
40  2026-02-05        oil b  City 1  108.5
96  2026-01-08     Staple A  City 1   99.5
rows: 97 expected 96
          date         item    city  price
40  2026-02-05        oil b  City 1  108.5
42  2026-02-05        Oil B  City 1  318.0
44  2026-02-05      Pulse C  City 1  202.5
46  2026-02-05  Sweetener D  City 1  161.5


## 2. Court trial: naive one-liners an AI often suggests (run on copies)
Each is tested to see whether it removes a valid observation or silently changes a label/number.

In [4]:
t1=raw.dropna()
print('dropna: rows',len(raw),'->',len(t1),'| removes valid Pulse C City 2 2026-01-01 row (its price is genuinely missing, the row is real)')
t2=raw.copy(); t2['item']=t2.item.str.title()
print('str.title: Oil B City 1 on 2026-02-05 now appears',((t2.item=='Oil B')&(t2.city=='City 1')&(t2.date=='2026-02-05')).sum(),'times (expected 1) -> silent relabel of a Staple A row')
t3=raw.copy(); t3['price']=pd.to_numeric(t3.price,errors='coerce')
print('to_numeric(coerce): new NaNs =',t3.price.isna().sum(),'vs 1 originally -> "1,250" silently erased')
t4=raw.drop_duplicates(subset=['date','item','city'])
print('drop_duplicates(subset=keys): rows',len(raw),'->',len(t4))
t5=raw.copy(); t5['p']=pd.to_numeric(raw.price.str.replace(',','',regex=False),errors='coerce'); t5=t5[t5.p<t5.p.quantile(.99)]
print('quantile trim: rows',len(raw),'->',len(t5),'(drops valid high-priced Oil B rows, not just the error)')

dropna: rows 97 -> 96 | removes valid Pulse C City 2 2026-01-01 row (its price is genuinely missing, the row is real)
str.title: Oil B City 1 on 2026-02-05 now appears 2 times (expected 1) -> silent relabel of a Staple A row
to_numeric(coerce): new NaNs = 2 vs 1 originally -> "1,250" silently erased
drop_duplicates(subset=keys): rows 97 -> 96
quantile trim: rows 97 -> 95 (drops valid high-priced Oil B rows, not just the error)


## 3. Cleaning on the copy, with a flag column so every change is visible

In [5]:
df=raw.copy()
df['row_id']=raw.index                     # original position, for traceability
df['flag']=''
def flag(mask,msg):
    df.loc[mask,'flag']=df.loc[mask,'flag'].apply(lambda s:(s+'; ' if s else '')+msg)

# C1 exact duplicate (row appended at end) -> drop
dup=df.duplicated(subset=['date','item','city','price','unit','data_label'],keep='first')
print('dropping duplicate row_ids:',df[dup].row_id.tolist()); df=df[~dup].copy()

# C2 price text -> number (remove thousands comma only); keep original text
df['price_raw']=df.price
df['price']=pd.to_numeric(df.price_raw.str.replace(',','',regex=False),errors='coerce')

# C3 mislabeled item: 'oil b' on 2026-02-05 City 1 (Oil B already present that day; Staple A City 1 missing that day; 108.5 fits Staple A)
m=(df.item=='oil b')&(df.date=='2026-02-05')&(df.city=='City 1')
df.loc[m,'item']='Staple A'; flag(m,'label corrected oil b -> Staple A (inferred, confirm with source)')

# C4 sentinel 9999 -> missing
m=df.price==9999; df.loc[m,'price']=float('nan'); flag(m,'9999 sentinel treated as missing')

# C5 '1,250': format fixed but implausible (~10x neighbours 105-122) -> excluded from price_clean, NOT corrected
m=df.price_raw=='1,250'; flag(m,'comma removed; 1250 implausible vs neighbours, excluded, not guessed')
df['price_clean']=df.price.where(~m)

# C6 genuine missing price -> retain row, no imputation
m=df.price_raw.isna(); flag(m,'price missing in source; retained, not imputed')
df=df.reset_index(drop=True)
df[df.flag!=''][['row_id','date','item','city','price_raw','price_clean','flag']]

dropping duplicate row_ids: [96]


,row_id,date,item,city,price_raw,price_clean,flag
5,5,2026-01-01,Pulse C,City 2,NaN,NaN,"price missing in source; retained, not imputed"
17,17,2026-01-15,Staple A,City 2,"1,250",NaN,"comma removed; 1250 implausible vs neighbours,..."
30,30,2026-01-22,Sweetener D,City 1,9999,NaN,9999 sentinel treated as missing
40,40,2026-02-05,Staple A,City 1,108.5,108.5,"label corrected oil b -> Staple A (inferred, c..."


## 4. Before / after table and expected-count check

In [6]:
def prof(d,col):
    return {'rows':len(d),'missing price':int(d[col].isna().sum()),'distinct items':d.item.nunique(),'max price':d[col].max(),'mean price':round(d[col].mean(),2)}
rawnum=pd.to_numeric(raw.price.str.replace(',','',regex=False),errors='coerce')
before=prof(raw.assign(p=rawnum),'p'); after=prof(df,'price_clean')
pd.DataFrame({'before':before,'after':after})

,before,after
rows,97.00,96.00
missing price,1.00,3.00
distinct items,5.00,4.00
max price,9999.00,350.00
mean price,315.46,203.61


In [7]:
print(df.groupby('item').price_clean.describe())
print(df.groupby(['item','city']).size().unstack())
full=pd.MultiIndex.from_product([sorted(raw.date.unique()),['Staple A','Oil B','Pulse C','Sweetener D'],['City 1','City 2']])
got=pd.MultiIndex.from_frame(df[['date','item','city']])
print('rows:',len(df),'| every date-item-city exactly once:',got.is_unique and set(got)==set(full))
print('missing price_clean:',df.price_clean.isna().sum(),'(3 expected: 1 source-missing, 1 sentinel, 1 excluded 1,250)')

             count        mean        std    min     25%    50%     75%    max
item                                                                          
Oil B         24.0  324.833333  14.529331  298.0  314.75  324.0  337.00  350.0
Pulse C       23.0  205.913043   3.912835  200.0  202.50  206.0  208.50  213.5
Staple A      23.0  111.260870   6.271997   99.5  106.75  112.5  115.25  122.0
Sweetener D   23.0  167.152174   9.695869  149.0  160.50  167.5  174.25  184.5
city         City 1  City 2
item                       
Oil B            12      12
Pulse C          12      12
Staple A         12      12
Sweetener D      12      12
rows: 96 | every date-item-city exactly once: True
missing price_clean: 3 (3 expected: 1 source-missing, 1 sentinel, 1 excluded 1,250)


In [8]:
df.to_csv('teaching_prices_clean.csv',index=False)
print('saved; raw file untouched, sha256 unchanged:',hashlib.sha256(open(path,'rb').read()).hexdigest()[:16])

saved; raw file untouched, sha256 unchanged: 5818dad3807487d4
